# Connect 4 — **ThompsonZero-C4** (Dirichlet state & action values)

A radical simplification of the chess ThompsonZero methodology, retargeted at
Connect 4 for fast benchmarking and proof of concept. Everything is implemented
in `connect4_dirichlet_utils.py`; this notebook is only a `Config`, one call, and
the plots.

## The network

A ResNet trunk feeding **two heads**, each emitting a Dirichlet over the 3-way
outcome `(win, draw, loss)` **from the mover's perspective**:

| head | output | belief |
|---|---|---|
| **state** | `p_win, p_draw, p_loss, c` | `V(s) = Dir(c · p)` |
| **action** | the same 4 numbers **per action** | `Q(s,a) = Dir(c_a · p_a)` |

That is the whole network — no policy head, no scalar value head, nothing
derived. The scalar used everywhere is `v = p_win − p_loss ∈ [−1, 1]`, which is
exactly Connect 4's own utility scale.

## The search

1. **Expand** a node → store `V(s)` and every `Q(s,a)`.
2. **Thompson-sample** every legal action: draw one `(x_w, x_d, x_l)` from that
   action's current belief and descend the argmax of `x_w − x_l`.
3. Repeat until an **unexpanded or terminal** edge is reached; expand it.
4. **Back up** exactly one Dirichlet — the freshly expanded leaf's own state
   belief `V(leaf)`, or a proven-terminal spike — flipping win↔loss at every ply.

Each node and each edge keeps a running **evidence accumulator** of everything
backed up through it, collapsed back to a single Dirichlet in `O(1)` from four
running sums `(n, Σm, Σm², Σv)`. Selection samples from

$$\alpha_{\text{sel}}(s,a) \;=\; \underbrace{\alpha_\theta(s,a)}_{\text{network prior}} \;+\; \underbrace{\alpha_{\text{obs}}(s,a)}_{\text{search evidence}}$$

so the network's belief is a *prior* whose pseudo-counts search adds evidence to.
After the simulation budget, **one more Thompson sample** at the root picks the
self-play move.

### Three ways to collapse the evidence

`search_agg` and `target_agg` pick this independently for the two roles, because
they want different things:

| rule | what it collapses | concentration after `n` observations of `Dir(α)` |
|---|---|---|
| `'mixture'` *(default)* | the mixture `(1/n) Σ Dir(αᵢ)` | stays at `α₀` — it measures **disagreement**, a property of the position |
| `'mean'` | the average draw `(1/n) Σ Xᵢ` | `n(α₀+1) − 1` — grows linearly with visits |
| `'sum'` | conjugate evidence `Σ αᵢ` | `n·α₀` — grows linearly, precision-weighted mean |

`'mixture'` everywhere is the default. Its trade-off is that concentration does
**not** grow with visits, so Thompson exploration does not anneal within a
search. `search_agg='mean'` is the natural fix; pairing it with
`target_agg='mixture'` gives an annealing search *and* a training target that
does not depend on the simulation budget. (With `'mean'`/`'sum'` targets, set
`kl_normalize=True` — their concentration scales with the playout cap.)

## The losses

| term | target |
|---|---|
| `klv` | `KL( Ô(s) ‖ V_θ(s) )` — observed state belief |
| `kla` | `KL( Ô(s,a) ‖ Q_θ(s,a) )` — observed action beliefs, scored only on edges search actually touched |
| `cev` | `CE( z , p_θ(s) )` — the game result, from `s`'s mover's view |
| `cea` | `CE( z , p_θ(s,a*) )` — same, on the move actually played |
| `cons` | `KL( flip(V_θ(s')) ‖ Q_θ(s,a*) )` — taking `a*` must be worth what the resulting state is worth, one ply flipped (state side stop-gradient) |

The cross-entropy terms ground everything in real results; the KL terms distil
search back into the net. An MCTS-Solver overlay proves terminal lines exactly
and emits extra exact-labelled samples.

## What carried over from the chess run

Every performance optimisation — multiprocess self-play workers behind one
central batched inference server, batched-leaf waves with virtual loss and leaf
de-duplication, `O(1)` incremental accumulators, exact-vs-Gaussian selection
sampling, subtree reuse, fp16 observations, on-device Dirichlet KL with
DirectML-friendly `lgamma`/`digamma`, `LerpFreeAdamW`, `torch.compile`,
checkpoint/resume — plus the whole logging and two-tier evaluation scheme.

Dropped: the policy-prior head and its KL, the Stockfish supervised bootstrap
(chess-only), and `z_mix` (the CE terms do that job directly). The endgame-first
curriculum and backward-restart pool are still available but **off** by default —
a Connect 4 game is at most 42 plies, so full self-play from move 1 is already
cheap and on-distribution.

> **Note on the observation tensor.** open_spiel's Connect 4 tensor is *not*
> perspective-relative — it ignores the `player` argument. The utils module
> detects that and builds a mover-relative (self-first) tensor itself, so the
> network can tell whose move it is.

## Reading the log line

```
ep   500 | loss 3.10 (klv 0.52 kla 0.51 cev 0.577 cea 0.748 cons 0.47)
         | sh klv 17% kla 33% cev 19% cea 24% cons 8%
         | conc(pred/tgt) v 0.4/40.2 a 0.4/13.5
         | dr 0% ply 14 buf 12k aux 90 | lr 1.77e-03 | vs 250 (no-MCTS) W12 D1 L7
         perf: 3.49 games/s | wait(sp) 33% train 38% | NNbatch 29 (43 fwd/s)
```

* **`sh`** — each term's *weighted* share of the total loss. This is what you
  tune `loss_weights` by: if one term sits near 100%, the others have stopped
  training.
* **`conc(pred/tgt)`** — predicted vs target Dirichlet concentration per head.
  A predicted value far below the target means the net is underfitting its own
  certainty.
* **`perf`** — `wait(sp)` high means self-play is the bottleneck; `train` high
  means the optimiser is; a small `NNbatch` means the GPU is underfed (raise
  `games_per_worker` or `selfplay_workers`).

In [ ]:
%pip install open_spiel -q
# Load the shared implementation module (the whole method lives there; this
# notebook is only configuration + plots).  Inside the repo it sits next to this
# notebook; on Colab we fetch it from the branch.
import os, sys, urllib.request
_NAME = 'connect4_dirichlet_utils.py'
_BRANCH = 'claude/connect4-dirichlet-values-my96dt'
_URL = ('https://raw.githubusercontent.com/calvinpozderac-claude/open_spiel/'
        f'{_BRANCH}/open_spiel/colabs/{_NAME}')

_path = next((p for p in (_NAME,
                          os.path.join('open_spiel', 'colabs', _NAME),
                          os.path.join('..', 'colabs', _NAME))
              if os.path.exists(p)), None)
if _path is None:
    urllib.request.urlretrieve(_URL, _NAME)
    _path = _NAME
sys.path.insert(0, os.path.dirname(os.path.abspath(_path)))

import importlib
import connect4_dirichlet_utils as c4
importlib.reload(c4)
print('loaded', c4.__file__)

In [ ]:
# ── The ONLY tunables. Every default is documented in Config's definition; ────
# ── anything not listed here keeps that default. ─────────────────────────────
cfg = c4.Config(
    # run
    num_episodes   = 20_000,
    channels       = 64,
    num_blocks     = 5,
    head_ch        = 16,
    checkpoint_dir = 'c4_dirichlet_ckpt',

    # evidence collapse: 'mixture' | 'mean' | 'sum', chosen independently for
    # search and for training targets (see the write-up above).
    search_agg  = 'mixture',
    target_agg  = 'mixture',
    kl_normalize = False,     # turn ON with 'mean'/'sum' targets, and raise the
                              # two KL weights when you do

    # search
    selection      = 'dirichlet',   # 'gaussian' = ~6x cheaper approximation
    fast_sims      = 100,           # 75% of games
    full_sims      = 400,           # the rest
    fast_prob      = 0.75,
    temp_threshold = 12,            # full-temperature Thompson moves for N plies
    late_temp      = 8.0,           # then sharpen (concentration x this)

    # self-play
    use_workers      = True,        # N processes + one batched inference server
    games_per_worker = 32,          # THE lever on GPU batch size
    worker_wave      = 8,
    pool_prob        = 0.15,        # frac. of games vs a frozen benchmark/random
    random_pool_frac = 0.5,

    # training — (klv, kla, cev, cea, cons)
    loss_weights       = (1.0, 2.0, 1.0, 1.0, 0.5),
    batch_size         = 512,
    train_steps_per_ep = 8,
    max_buffer         = 150_000,
    lr_peak            = 2e-3,
    lr_decay_eps       = 8_000,

    # eval — QUICK: search-free pulse vs the last checkpoint.
    #        DEEP:  add a checkpoint to the MCTS running-Elo pool.
    quick_eval_every = 500,
    quick_eval_games = 40,
    deep_eval_every  = 2_500,
    eval_sims        = 128,
)
cfg

In [ ]:
hist = c4.run_training(cfg)

In [ ]:
c4.plot_history(hist);

## Arena

Pit two saved benchmarks head-to-head at chosen search budgets, or measure **how
much search adds** by giving one side `sims=0` (search-free). A label may also be
`'random'`.

In [ ]:
for a, b, sa, sb in [('2500', 'random', 0, 0),
                    ('2500', '2500', 128, 0),
                    ('5000', '2500', 128, 128)]:
    try:
        w, d, l = c4.duel(cfg, a, b, sims_a=sa, sims_b=sb, n_games=20)
        print(f'{a}@{sa or "no-MCTS"} vs {b}@{sb or "no-MCTS"}: W{w} D{d} L{l}')
    except FileNotFoundError as e:
        print(f'skip {a} vs {b}: {e.filename} not saved yet')

## Self-tests

Validates the Dirichlet moment matching against Monte Carlo, the incremental
accumulators against a from-scratch recompute, backup perspective flipping,
solver propagation, the loss shapes and masking, and an end-to-end Connect 4
self-play episode.

In [ ]:
import subprocess, os
_t = os.path.join(os.path.dirname(c4.__file__), 'connect4_dirichlet_tests.py')
print(subprocess.run([sys.executable, _t], cwd=os.path.dirname(_t),
                     capture_output=True, text=True).stdout[-3000:])